In [ ]:

import contextlib

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from plotly.subplots import make_subplots
from torch import Tensor
from tqdm.autonotebook import tqdm

from src.config.base import BaseConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import TimeConditioningConfig

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
USE_CUDA = DEVICE.startswith("cuda")
AMP_DTYPE = torch.bfloat16

torch.set_float32_matmul_precision("high")


@contextlib.contextmanager
def training_device_context():
    if USE_CUDA:
        with (
            torch.cuda.device(DEVICE),
            torch.device(DEVICE),
            torch.autocast(device_type="cuda", dtype=AMP_DTYPE),
        ):
            yield
        return
    with torch.device(DEVICE):
        yield


print(f"device: {DEVICE}")


In [ ]:

class BimodalDataConfig(BaseConfig):
    offset: Float[Tensor, "2"]
    center: float
    std: float
    scale: float
    std_cap: float

    def _sample_capped_noise(
        self,
        batch_size: int,
    ) -> Float[Tensor, "batch 2"]:
        z = torch.randn((batch_size, 2))
        norms = z.norm(dim=1, keepdim=True).clamp_min(1e-6)
        capped_norms = norms.clamp_max(self.std_cap)
        return z * (self.std * capped_norms / norms)

    def sample_mode(
        self,
        batch_size: int,
        mode_id: int,
    ) -> Float[Tensor, "batch 2"]:
        x = self._sample_capped_noise(batch_size)
        direction = 1.0 if mode_id == 0 else -1.0
        x[:, 0] += direction * self.center / self.scale
        return x * self.scale + self.offset

    def sample_with_mode_ids(
        self,
        batch_size: int,
    ) -> tuple[Float[Tensor, "batch 2"], Tensor]:
        if batch_size % 2 != 0:
            raise ValueError("batch_size must be even")
        per_mode_size = batch_size // 2
        mode_ids = torch.cat(
            [
                torch.zeros(per_mode_size, dtype=torch.long),
                torch.ones(per_mode_size, dtype=torch.long),
            ]
        )
        x = torch.cat(
            [
                self.sample_mode(per_mode_size, mode_id=0),
                self.sample_mode(per_mode_size, mode_id=1),
            ],
            dim=0,
        )
        return x, mode_ids

    def sample_prior(
        self,
        batch_size: int,
    ) -> Float[Tensor, "batch 2"]:
        return torch.randn((batch_size, 2))

    def sample_contours(
        self,
        per_contour_size: int,
        zscores: tuple[float, ...],
    ) -> tuple[Float[Tensor, "batch 2"], Tensor, tuple[float, ...]]:
        angles = torch.linspace(0.0, 2.0 * torch.pi, per_contour_size + 1)[:-1]
        unit_circle = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
        contour_points = []
        contour_ids = []
        for contour_id, zscore in enumerate(zscores):
            contour_points.append(zscore * unit_circle)
            contour_ids.append(
                torch.full((per_contour_size,), contour_id, dtype=torch.long)
            )
        return (
            torch.cat(contour_points, dim=0),
            torch.cat(contour_ids, dim=0),
            zscores,
        )


class DirectTransportExperimentConfig(BaseConfig):
    num_particles: int
    train_batch_size: int
    warmup_steps: int
    total_steps: int
    lr: float
    weight_decay: float
    grad_clip_norm: float
    min_t: float
    max_t: float
    transport_step_size: float
    max_transport_update_norm: float
    log_every_steps: int
    snapshot_every_steps: int
    scatter_num_samples: int
    vector_num_samples: int
    contour_levels: tuple[float, ...]
    contour_points_per_level: int
    contour_grid_resolution: int
    vector_grid_resolution: int
    drift_grid_resolution: int
    drift_noise_draws: int
    plot_t_values: list[float]


data_config = BimodalDataConfig(
    offset=torch.tensor([0.0, 0.0]),
    center=3.0,
    std=0.2,
    scale=2.0,
    std_cap=2.0,
)

experiment_config = DirectTransportExperimentConfig(
    num_particles=1024 * 1024,
    train_batch_size=8192,
    warmup_steps=1000,
    total_steps=16000,
    lr=1e-3,
    weight_decay=1e-2,
    grad_clip_norm=1.0,
    min_t=0.025,
    max_t=0.975,
    transport_step_size=0.08,
    max_transport_update_norm=1.0,
    log_every_steps=50,
    snapshot_every_steps=1000,
    scatter_num_samples=4096,
    vector_num_samples=96,
    contour_levels=(0.1, 0.5, 1.0, 1.5),
    contour_points_per_level=160,
    contour_grid_resolution=144,
    vector_grid_resolution=34,
    drift_grid_resolution=28,
    drift_noise_draws=6,
    plot_t_values=[0.05, 0.15, 0.3, 0.5, 0.7, 0.9],
)

experiment_config


In [ ]:

class DirectTransportModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.noise_critic = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 2],
            time_conditioning_config=TimeConditioningConfig(
                min_t_lambda=5e-3,
                max_t_lambda=1.0,
                sinusoidal_dim=512,
                hidden_dim=512,
                output_dim=512,
            ),
        ).get_model()

    def implied_denoiser(
        self,
        y_t: Float[Tensor, "batch 2"],
        t: Float[Tensor, "batch 1"],
    ) -> Float[Tensor, "batch 2"]:
        predicted_noise = self.noise_critic(y_t, t=t)
        return implied_denoiser_from_noise_prediction(
            y_t=y_t,
            predicted_noise=predicted_noise,
            t=t,
        )

    def get_optimizer(self) -> torch.optim.Optimizer:
        return torch.optim.AdamW(
            self.noise_critic.parameters(),
            lr=experiment_config.lr,
            weight_decay=experiment_config.weight_decay,
        )


model = DirectTransportModel().to(DEVICE)
optimizer = model.get_optimizer()

transport_points, transport_mode_ids = data_config.sample_with_mode_ids(
    experiment_config.num_particles,
)
transport_points = transport_points.to(DEVICE)
transport_mode_ids = transport_mode_ids.to(DEVICE)
initial_transport_points = transport_points.detach().clone()

logs = {
    "steps": [],
    "score_matching": [],
    "transport_update_norm": [],
    "mean_error": [],
    "cov_error": [],
}

print(f"transport points: {transport_points.shape[0]:,}")


In [ ]:

MODE_COLORS = ("#1f77b4", "#ff7f0e")
CONTOUR_COLORS = ("#440154", "#3b528b", "#21918c", "#5ec962", "#fde725")
METRIC_COLORS = {
    "score_matching": "#d62728",
    "transport_update_norm": "#2ca02c",
    "mean_error": "#1f77b4",
    "cov_error": "#9467bd",
}


def current_training_step() -> int:
    if len(logs["steps"]) == 0:
        return 0
    return int(logs["steps"][-1])


def subsample_indices(total_size: int, max_points: int) -> Tensor:
    if total_size <= max_points:
        return torch.arange(total_size)
    return torch.linspace(0, total_size - 1, max_points).round().long()


def normalize_vectors(
    vectors: Float[Tensor, "batch 2"],
    display_length: float,
) -> tuple[Float[Tensor, "batch 2"], Float[Tensor, "batch"]]:
    magnitudes = vectors.norm(dim=1)
    normalized = vectors / magnitudes.unsqueeze(-1).clamp_min(1e-6)
    return display_length * normalized, magnitudes


def regular_grid_display_length(
    xs: Tensor,
    ys: Tensor,
    fraction: float,
) -> float:
    if xs.numel() < 2 or ys.numel() < 2:
        return fraction
    x_step = float((xs[1] - xs[0]).item())
    y_step = float((ys[1] - ys[0]).item())
    return fraction * min(abs(x_step), abs(y_step))


def square_grid_limits(
    points: Float[Tensor, "batch 2"],
    padding: float,
) -> tuple[float, float, float, float]:
    mins = points.min(dim=0).values
    maxs = points.max(dim=0).values
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float((maxs - mins).max().item())
    radius = max(radius * (1.0 + padding), 1.0)
    return (
        float(center[0] - radius),
        float(center[0] + radius),
        float(center[1] - radius),
        float(center[1] + radius),
    )


def build_square_grid(
    points: Float[Tensor, "batch 2"],
    resolution: int,
) -> tuple[Float[Tensor, "grid 2"], Tensor, Tensor]:
    x_min, x_max, y_min, y_max = square_grid_limits(points, padding=0.12)
    xs = torch.linspace(
        x_min,
        x_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    ys = torch.linspace(
        y_min,
        y_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
    grid_points = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=-1)
    return grid_points, xs.detach().cpu(), ys.detach().cpu()


def add_quiver_traces(
    fig: go.Figure,
    *,
    points: Float[Tensor, "batch 2"],
    vectors: Float[Tensor, "batch 2"],
    magnitudes: Float[Tensor, "batch"],
    color: str,
    name: str,
    row: int,
    col: int,
    visible: bool,
    showlegend: bool,
    show_hover: bool,
    arrow_scale: float,
    line_width: float,
    opacity: float,
) -> None:
    quiver = ff.create_quiver(
        x=points[:, 0].detach().cpu().float().tolist(),
        y=points[:, 1].detach().cpu().float().tolist(),
        u=vectors[:, 0].detach().cpu().float().tolist(),
        v=vectors[:, 1].detach().cpu().float().tolist(),
        scale=1.0,
        arrow_scale=arrow_scale,
        line_color=color,
        name=name,
    )
    for trace_id, trace in enumerate(quiver.data):
        trace.visible = visible
        trace.showlegend = showlegend and trace_id == 0
        trace.name = name
        trace.hoverinfo = "skip"
        trace.line.width = line_width
        trace.opacity = opacity
        fig.add_trace(trace, row=row, col=col)

    if not show_hover:
        return

    hover_text = [
        f"{name}<br>|v|={float(magnitude):.4f}"
        for magnitude in magnitudes.detach().cpu().float().tolist()
    ]
    fig.add_trace(
        go.Scatter(
            x=points[:, 0].detach().cpu().float().tolist(),
            y=points[:, 1].detach().cpu().float().tolist(),
            mode="markers",
            marker={"size": 10, "color": color, "opacity": 0.001},
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
            name=name,
            visible=visible,
        ),
        row=row,
        col=col,
    )


def add_model_and_gaussian_score_hover_trace(
    fig: go.Figure,
    *,
    points: Float[Tensor, "batch 2"],
    model_score_magnitudes: Float[Tensor, "batch"],
    gaussian_score_magnitudes: Float[Tensor, "batch"],
    row: int,
    col: int,
    visible: bool,
) -> None:
    hover_text = [
        (
            f"model score field<br>|v_model|={float(model_magnitude):.4f}<br>"
            f"analytic Gaussian score field<br>|v_gaussian|={float(gaussian_magnitude):.4f}"
        )
        for model_magnitude, gaussian_magnitude in zip(
            model_score_magnitudes.detach().cpu().float().tolist(),
            gaussian_score_magnitudes.detach().cpu().float().tolist(),
            strict=True,
        )
    ]
    fig.add_trace(
        go.Scatter(
            x=points[:, 0].detach().cpu().float().tolist(),
            y=points[:, 1].detach().cpu().float().tolist(),
            mode="markers",
            marker={"size": 12, "color": "#111111", "opacity": 0.001},
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
            name="model / analytic Gaussian score magnitudes",
            visible=visible,
        ),
        row=row,
        col=col,
    )


def add_latent_contour_traces(
    fig: go.Figure,
    *,
    contour_points: Float[Tensor, "batch 2"],
    contour_ids: Tensor,
    contour_levels: tuple[float, ...],
    row: int,
    col: int,
    visible: bool,
    showlegend: bool,
) -> None:
    for contour_id, contour_level in enumerate(contour_levels):
        mask = contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=contour_points[mask, 0].detach().cpu().numpy(),
                y=contour_points[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 2.0,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                    "dash": "dot",
                },
                name=f"z = {contour_level}",
                visible=visible,
                showlegend=showlegend,
            ),
            row=row,
            col=col,
        )


def add_contour_trace(
    fig: go.Figure,
    *,
    xs: Tensor,
    ys: Tensor,
    values: Float[Tensor, "y x"],
    row: int,
    col: int,
    visible: bool,
    name: str,
    colorbar_title: str,
    hover_label: str,
) -> None:
    fig.add_trace(
        go.Contour(
            x=xs.numpy(),
            y=ys.numpy(),
            z=values.detach().cpu().numpy(),
            colorscale="Viridis",
            ncontours=18,
            contours={"coloring": "heatmap", "showlabels": False},
            line={"width": 0.8},
            opacity=0.38,
            showscale=False,
            hovertemplate=(
                "x=%{x:.2f}<br>y=%{y:.2f}<br>"
                f"{hover_label}=%{{z:.4f}}<extra></extra>"
            ),
            name=name,
            visible=visible,
        ),
        row=row,
        col=col,
    )


def transport_moment_errors(
    points: Float[Tensor, "batch 2"],
) -> tuple[float, float]:
    with torch.no_grad():
        mean = points.mean(dim=0)
        centered = points - mean.unsqueeze(0)
        covariance = centered.T @ centered / max(points.shape[0] - 1, 1)
        mean_error = float(mean.norm().item())
        covariance_error = float(
            (covariance - torch.eye(2, device=points.device, dtype=points.dtype))
            .norm()
            .item()
        )
    return mean_error, covariance_error


def plot_training_metrics() -> go.Figure:
    step = current_training_step()
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            "optimization signals",
            "Gaussian moment errors",
        ],
        horizontal_spacing=0.12,
    )
    fig.add_trace(
        go.Scatter(
            x=logs["steps"],
            y=logs["score_matching"],
            mode="lines",
            line={"width": 3, "color": METRIC_COLORS["score_matching"]},
            name="score matching",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=logs["steps"],
            y=logs["transport_update_norm"],
            mode="lines",
            line={"width": 3, "color": METRIC_COLORS["transport_update_norm"]},
            name="transport update norm",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=logs["steps"],
            y=logs["mean_error"],
            mode="lines",
            line={"width": 3, "color": METRIC_COLORS["mean_error"]},
            name="mean error",
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=logs["steps"],
            y=logs["cov_error"],
            mode="lines",
            line={"width": 3, "color": METRIC_COLORS["cov_error"]},
            name="covariance error",
        ),
        row=1,
        col=2,
    )
    fig.update_xaxes(title_text="optimization step", showgrid=True, zeroline=False)
    fig.update_yaxes(showgrid=True, zeroline=False)
    fig.update_layout(
        title=f"transport training metrics at step {step}",
        height=360,
        width=1120,
        margin={"l": 40, "r": 20, "t": 80, "b": 40},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
    )
    return fig


def plot_transport_state(
    scatter_num_samples: int,
) -> go.Figure:
    step = current_training_step()
    subset_indices = subsample_indices(
        transport_points.shape[0],
        scatter_num_samples,
    ).to(transport_points.device)
    contour_points, contour_ids, contour_levels = data_config.sample_contours(
        per_contour_size=experiment_config.contour_points_per_level,
        zscores=experiment_config.contour_levels,
    )
    with torch.no_grad():
        current_points = transport_points[subset_indices].detach().cpu()
        initial_points = initial_transport_points[subset_indices].detach().cpu()
        mode_ids = transport_mode_ids[subset_indices].detach().cpu()
        target_points = data_config.sample_prior(scatter_num_samples).cpu()

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            "transport particles vs Gaussian target",
            "initial support vs transported particles",
        ],
        horizontal_spacing=0.08,
    )
    fig.add_trace(
        go.Scatter(
            x=target_points[:, 0].numpy(),
            y=target_points[:, 1].numpy(),
            mode="markers",
            marker={"size": 4, "color": "#111111", "opacity": 0.12},
            name="target Gaussian",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=initial_points[:, 0].numpy(),
            y=initial_points[:, 1].numpy(),
            mode="markers",
            marker={"size": 4, "color": "#111111", "opacity": 0.12},
            name="initial transport cloud",
        ),
        row=1,
        col=2,
    )
    for mode_id, color in enumerate(MODE_COLORS):
        mask = mode_ids == mode_id
        fig.add_trace(
            go.Scatter(
                x=current_points[mask, 0].numpy(),
                y=current_points[mask, 1].numpy(),
                mode="markers",
                marker={"size": 4, "color": color, "opacity": 0.26},
                name=f"transport mode {mode_id}",
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=current_points[mask, 0].numpy(),
                y=current_points[mask, 1].numpy(),
                mode="markers",
                marker={"size": 4, "color": color, "opacity": 0.26},
                name=f"transport mode {mode_id}",
                showlegend=False,
            ),
            row=1,
            col=2,
        )
    add_latent_contour_traces(
        fig,
        contour_points=contour_points,
        contour_ids=contour_ids,
        contour_levels=contour_levels,
        row=1,
        col=1,
        visible=True,
        showlegend=True,
    )
    add_latent_contour_traces(
        fig,
        contour_points=contour_points,
        contour_ids=contour_ids,
        contour_levels=contour_levels,
        row=1,
        col=2,
        visible=True,
        showlegend=False,
    )
    for col in [1, 2]:
        fig.update_xaxes(
            showgrid=True,
            zeroline=False,
            scaleanchor=f"y{col if col > 1 else ''}",
            scaleratio=1,
            row=1,
            col=col,
        )
        fig.update_yaxes(showgrid=True, zeroline=False, row=1, col=col)
    fig.update_layout(
        title=f"transport state at step {step}",
        height=430,
        width=1360,
        margin={"l": 30, "r": 20, "t": 80, "b": 30},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
    )
    return fig


In [ ]:

def implied_denoiser_from_noise_prediction(
    y_t: Float[Tensor, "batch 2"],
    predicted_noise: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return (y_t - t * predicted_noise) / (1.0 - t).clamp_min(1e-4)


def analytic_gaussian_implied_denoiser(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    variance = (1.0 - t).square() + t.square()
    return ((1.0 - t) / variance) * y_t


def score_from_noise_prediction(
    predicted_noise: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return -predicted_noise / t.clamp_min(1e-4)


def score_from_implied_denoiser(
    y_t: Float[Tensor, "batch 2"],
    implied_denoiser: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return ((1.0 - t) * implied_denoiser - y_t) / t.square().clamp_min(1e-6)


def evaluate_model_score(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    predicted_noise = model.noise_critic(y_t, t=t)
    return score_from_noise_prediction(predicted_noise, t=t)


def evaluate_gaussian_score(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    analytic_denoiser = analytic_gaussian_implied_denoiser(y_t, t=t)
    return score_from_implied_denoiser(
        y_t=y_t,
        implied_denoiser=analytic_denoiser,
        t=t,
    )


def evaluate_score_difference(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return evaluate_model_score(y_t, t=t) - evaluate_gaussian_score(y_t, t=t)


def sample_noise_levels(
    batch_size: int,
    *,
    device: str,
    dtype: torch.dtype,
) -> Float[Tensor, "batch 1"]:
    t = torch.rand((batch_size, 1), device=device, dtype=dtype)
    return t * (experiment_config.max_t - experiment_config.min_t) + experiment_config.min_t


def clip_vector_norms(
    vectors: Float[Tensor, "batch 2"],
    max_norm: float,
) -> tuple[Float[Tensor, "batch 2"], Float[Tensor, "batch"]]:
    norms = vectors.norm(dim=1, keepdim=True).clamp_min(1e-6)
    scale = torch.clamp(max_norm / norms, max=1.0)
    return vectors * scale, norms.squeeze(-1)


def implied_denoiser_mismatch_gradients(
    clean_latents: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
    noise: Float[Tensor, "batch 2"],
) -> tuple[
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
]:
    noisy_latents = ((1.0 - t) * clean_latents + t * noise).detach().clone()
    noisy_latents.requires_grad_(True)
    predicted_noise = model.noise_critic(noisy_latents, t=t)
    implied_denoiser = implied_denoiser_from_noise_prediction(
        y_t=noisy_latents,
        predicted_noise=predicted_noise,
        t=t,
    )
    analytic_denoiser = analytic_gaussian_implied_denoiser(y_t=noisy_latents, t=t)
    mismatch = (implied_denoiser - analytic_denoiser).square().mean(dim=1)
    noisy_latent_drift = -torch.autograd.grad(mismatch.sum(), noisy_latents)[0]
    clean_latent_drift = (1.0 - t) * noisy_latent_drift
    return (
        noisy_latents.detach(),
        noisy_latent_drift.detach(),
        clean_latent_drift.detach(),
    )


def score_difference_transport_vectors(
    clean_latents: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
    noise: Float[Tensor, "batch 2"],
) -> tuple[
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
]:
    noisy_latents = ((1.0 - t) * clean_latents + t * noise).detach()
    noisy_transport = -evaluate_score_difference(noisy_latents, t=t)
    clean_transport = (1.0 - t) * noisy_transport
    return (
        noisy_latents.detach(),
        noisy_transport.detach(),
        clean_transport.detach(),
    )


def expected_clean_transport_field(
    clean_points: Float[Tensor, "batch 2"],
    noise_level: float,
    num_noise_draws: int,
) -> Float[Tensor, "batch 2"]:
    total_field = torch.zeros_like(clean_points)
    t = torch.full(
        (clean_points.shape[0], 1),
        float(noise_level),
        device=clean_points.device,
        dtype=clean_points.dtype,
    )
    with torch.no_grad():
        for _ in range(num_noise_draws):
            noise = torch.randn_like(clean_points)
            noisy_points = (1.0 - t) * clean_points + t * noise
            total_field = total_field - (1.0 - t) * evaluate_score_difference(
                noisy_points,
                t=t,
            )
    return total_field / num_noise_draws


def estimate_noise_prediction_loss(
    clean_latents: Float[Tensor, "batch 2"],
    noise_level: float,
    num_noise_draws: int,
) -> float:
    t = torch.full(
        (clean_latents.shape[0], 1),
        float(noise_level),
        device=clean_latents.device,
        dtype=clean_latents.dtype,
    )
    total_loss = clean_latents.new_tensor(0.0)
    with torch.no_grad():
        for _ in range(num_noise_draws):
            noise = torch.randn_like(clean_latents)
            noisy_latents = (1.0 - t) * clean_latents + t * noise
            predicted_noise = model.noise_critic(noisy_latents, t=t)
            total_loss = total_loss + F.mse_loss(predicted_noise, noise)
    return float((total_loss / num_noise_draws).item())


def plot_score_estimates() -> go.Figure:
    step = current_training_step()
    model_was_training = model.training
    model.eval()

    subset_indices = subsample_indices(
        transport_points.shape[0],
        experiment_config.scatter_num_samples,
    ).to(transport_points.device)
    clean_latents = transport_points[subset_indices].detach()
    mode_ids = transport_mode_ids[subset_indices].detach()
    contour_points, contour_ids, contour_levels = data_config.sample_contours(
        per_contour_size=experiment_config.contour_points_per_level,
        zscores=experiment_config.contour_levels,
    )

    snapshots = []
    for noise_level in experiment_config.plot_t_values:
        t = torch.full(
            (clean_latents.shape[0], 1),
            float(noise_level),
            device=clean_latents.device,
            dtype=clean_latents.dtype,
        )
        noise = torch.randn_like(clean_latents)
        with torch.no_grad():
            noisy_latents = (1.0 - t) * clean_latents + t * noise
            predicted_noise = model.noise_critic(noisy_latents, t=t)
            score_vectors = score_from_noise_prediction(predicted_noise, t=t)

            noisy_grid_points, noisy_xs, noisy_ys = build_square_grid(
                noisy_latents,
                resolution=experiment_config.contour_grid_resolution,
            )
            noisy_vector_display_length = regular_grid_display_length(
                noisy_xs,
                noisy_ys,
                fraction=1.15,
            )
            noisy_grid_t = torch.full(
                (noisy_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            noisy_score_difference = evaluate_score_difference(
                noisy_grid_points,
                t=noisy_grid_t,
            )
            noisy_mismatch_norm = noisy_score_difference.norm(dim=1).reshape(
                experiment_config.contour_grid_resolution,
                experiment_config.contour_grid_resolution,
            )

            clean_grid_points, clean_xs, clean_ys = build_square_grid(
                clean_latents,
                resolution=experiment_config.contour_grid_resolution,
            )
            clean_vector_display_length = regular_grid_display_length(
                clean_xs,
                clean_ys,
                fraction=1.15,
            )
            clean_grid_t = torch.full(
                (clean_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            pulled_noisy_grid_points = (1.0 - clean_grid_t) * clean_grid_points
            clean_pullback_score_difference = (
                1.0 - clean_grid_t
            ) * evaluate_score_difference(
                pulled_noisy_grid_points,
                t=clean_grid_t,
            )
            clean_pullback_mismatch_norm = clean_pullback_score_difference.norm(
                dim=1
            ).reshape(
                experiment_config.contour_grid_resolution,
                experiment_config.contour_grid_resolution,
            )

            noisy_vector_grid_points, _, _ = build_square_grid(
                noisy_latents,
                resolution=experiment_config.vector_grid_resolution,
            )
            noisy_vector_t = torch.full(
                (noisy_vector_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            noisy_grid_model_score = evaluate_model_score(
                noisy_vector_grid_points,
                t=noisy_vector_t,
            )
            noisy_grid_gaussian_score = evaluate_gaussian_score(
                noisy_vector_grid_points,
                t=noisy_vector_t,
            )

            clean_vector_grid_points, _, _ = build_square_grid(
                clean_latents,
                resolution=experiment_config.vector_grid_resolution,
            )
            clean_vector_t = torch.full(
                (clean_vector_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            clean_grid_score_difference = (
                1.0 - clean_vector_t
            ) * evaluate_score_difference(
                (1.0 - clean_vector_t) * clean_vector_grid_points,
                t=clean_vector_t,
            )

        noisy_drift_points, noisy_drift_vectors, _ = (
            implied_denoiser_mismatch_gradients(
                clean_latents=clean_latents,
                t=t,
                noise=noise,
            )
        )
        noisy_transport_points, noisy_transport_vectors, clean_transport_vectors = (
            score_difference_transport_vectors(
                clean_latents=clean_latents,
                t=t,
                noise=noise,
            )
        )

        drift_grid_points, drift_xs, drift_ys = build_square_grid(
            clean_latents,
            resolution=experiment_config.drift_grid_resolution,
        )
        drift_vector_display_length = regular_grid_display_length(
            drift_xs,
            drift_ys,
            fraction=1.05,
        )
        clean_drift_grid = expected_clean_transport_field(
            clean_points=drift_grid_points,
            noise_level=noise_level,
            num_noise_draws=experiment_config.drift_noise_draws,
        )
        drift_display_vectors, drift_magnitudes = normalize_vectors(
            clean_drift_grid.detach().cpu(),
            drift_vector_display_length,
        )
        clean_drift_norm = clean_drift_grid.norm(dim=1).reshape(
            experiment_config.drift_grid_resolution,
            experiment_config.drift_grid_resolution,
        )

        score_indices = subsample_indices(
            noisy_latents.shape[0],
            experiment_config.vector_num_samples,
        ).to(clean_latents.device)
        noisy_drift_indices = subsample_indices(
            noisy_drift_points.shape[0],
            experiment_config.vector_num_samples,
        ).to(clean_latents.device)
        clean_drift_indices = subsample_indices(
            clean_latents.shape[0],
            experiment_config.vector_num_samples,
        ).to(clean_latents.device)
        noisy_transport_indices = subsample_indices(
            noisy_transport_points.shape[0],
            experiment_config.vector_num_samples,
        ).to(clean_latents.device)
        clean_transport_indices = subsample_indices(
            clean_latents.shape[0],
            experiment_config.vector_num_samples,
        ).to(clean_latents.device)

        score_display_vectors, score_magnitudes = normalize_vectors(
            score_vectors[score_indices].detach().cpu(),
            0.32,
        )
        noisy_drift_display_vectors, noisy_drift_magnitudes = normalize_vectors(
            noisy_drift_vectors[noisy_drift_indices].detach().cpu(),
            0.24,
        )
        noisy_transport_display_vectors, noisy_transport_magnitudes = normalize_vectors(
            noisy_transport_vectors[noisy_transport_indices].detach().cpu(),
            0.24,
        )
        clean_drift_display_vectors, clean_drift_magnitudes = normalize_vectors(
            clean_transport_vectors[clean_transport_indices].detach().cpu(),
            0.22,
        )
        noisy_grid_model_score_display, noisy_grid_model_score_magnitudes = (
            normalize_vectors(
                noisy_grid_model_score.detach().cpu(),
                noisy_vector_display_length,
            )
        )
        noisy_grid_gaussian_score_display, noisy_grid_gaussian_score_magnitudes = (
            normalize_vectors(
                noisy_grid_gaussian_score.detach().cpu(),
                noisy_vector_display_length,
            )
        )
        clean_grid_score_difference_display, clean_grid_score_difference_magnitudes = (
            normalize_vectors(
                (-clean_grid_score_difference).detach().cpu(),
                clean_vector_display_length,
            )
        )

        snapshots.append(
            {
                "noise_level": noise_level,
                "noisy_latents": noisy_latents.detach().cpu(),
                "score_points": noisy_latents[score_indices].detach().cpu(),
                "score_vectors": score_display_vectors,
                "score_magnitudes": score_magnitudes,
                "noisy_drift_points": noisy_drift_points[noisy_drift_indices].detach().cpu(),
                "noisy_drift_vectors": noisy_drift_display_vectors,
                "noisy_drift_magnitudes": noisy_drift_magnitudes,
                "noisy_transport_points": noisy_transport_points[noisy_transport_indices].detach().cpu(),
                "noisy_transport_vectors": noisy_transport_display_vectors,
                "noisy_transport_magnitudes": noisy_transport_magnitudes,
                "clean_latents": clean_latents.detach().cpu(),
                "clean_drift_points": clean_latents[clean_transport_indices].detach().cpu(),
                "clean_drift_vectors": clean_drift_display_vectors,
                "clean_drift_magnitudes": clean_drift_magnitudes,
                "noisy_contour_xs": noisy_xs,
                "noisy_contour_ys": noisy_ys,
                "noisy_mismatch_norm": noisy_mismatch_norm.detach().cpu(),
                "clean_contour_xs": clean_xs,
                "clean_contour_ys": clean_ys,
                "clean_pullback_mismatch_norm": clean_pullback_mismatch_norm.detach().cpu(),
                "noisy_grid_score_points": noisy_vector_grid_points.detach().cpu(),
                "noisy_grid_score_vectors": noisy_grid_model_score_display,
                "noisy_grid_score_magnitudes": noisy_grid_model_score_magnitudes,
                "noisy_grid_gaussian_score_vectors": noisy_grid_gaussian_score_display,
                "noisy_grid_gaussian_score_magnitudes": noisy_grid_gaussian_score_magnitudes,
                "clean_grid_difference_points": clean_vector_grid_points.detach().cpu(),
                "clean_grid_difference_vectors": clean_grid_score_difference_display,
                "clean_grid_difference_magnitudes": clean_grid_score_difference_magnitudes,
                "drift_grid_points": drift_grid_points.detach().cpu(),
                "drift_grid_vectors": drift_display_vectors,
                "drift_grid_magnitudes": drift_magnitudes,
                "drift_xs": drift_xs,
                "drift_ys": drift_ys,
                "clean_drift_norm": clean_drift_norm.detach().cpu(),
                "mode_ids": mode_ids.detach().cpu(),
                "score_mode_ids": mode_ids[score_indices].detach().cpu(),
                "noisy_drift_mode_ids": mode_ids[noisy_drift_indices].detach().cpu(),
                "noisy_transport_mode_ids": mode_ids[noisy_transport_indices].detach().cpu(),
                "clean_drift_mode_ids": mode_ids[clean_transport_indices].detach().cpu(),
                "loss_estimate": estimate_noise_prediction_loss(
                    clean_latents=clean_latents,
                    noise_level=noise_level,
                    num_noise_draws=4,
                ),
            }
        )

    if model_was_training:
        model.train()

    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=[
            "noise-critic score estimate",
            "noisy mismatch-gradient + transport overlay",
            "clean-latent pullback transport field",
            "score mismatch norm + model / Gaussian score fields",
            "clean pullback transport norm + transport field",
            "noise-averaged clean transport norm + field",
        ],
        horizontal_spacing=0.06,
        vertical_spacing=0.12,
    )

    toggle_trace_indices = []
    for snapshot_index, snapshot in enumerate(snapshots):
        visible = snapshot_index == 0
        start_trace = len(fig.data)
        for mode_id, color in enumerate(MODE_COLORS):
            point_mask = snapshot["mode_ids"] == mode_id
            score_mask = snapshot["score_mode_ids"] == mode_id
            noisy_transport_mask = snapshot["noisy_transport_mode_ids"] == mode_id
            clean_drift_mask = snapshot["clean_drift_mode_ids"] == mode_id
            fig.add_trace(
                go.Scatter(
                    x=snapshot["noisy_latents"][point_mask, 0].numpy(),
                    y=snapshot["noisy_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"noisy latent mode {mode_id}",
                    visible=visible,
                    showlegend=visible,
                ),
                row=1,
                col=1,
            )
            if int(score_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["score_points"][score_mask],
                    vectors=snapshot["score_vectors"][score_mask],
                    magnitudes=snapshot["score_magnitudes"][score_mask],
                    color=color,
                    name=f"score mode {mode_id}",
                    row=1,
                    col=1,
                    visible=visible,
                    showlegend=visible,
                    show_hover=True,
                    arrow_scale=0.32,
                    line_width=1.2,
                    opacity=0.65,
                )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["noisy_latents"][point_mask, 0].numpy(),
                    y=snapshot["noisy_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"noisy mismatch mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=1,
                col=2,
            )
            if int(noisy_transport_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["noisy_transport_points"][noisy_transport_mask],
                    vectors=snapshot["noisy_transport_vectors"][noisy_transport_mask],
                    magnitudes=snapshot["noisy_transport_magnitudes"][noisy_transport_mask],
                    color=color,
                    name=f"noisy transport mode {mode_id}",
                    row=1,
                    col=2,
                    visible=visible,
                    showlegend=False,
                    show_hover=True,
                    arrow_scale=0.32,
                    line_width=1.2,
                    opacity=0.65,
                )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["clean_latents"][point_mask, 0].numpy(),
                    y=snapshot["clean_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"clean latent mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=1,
                col=3,
            )
            if int(clean_drift_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["clean_drift_points"][clean_drift_mask],
                    vectors=snapshot["clean_drift_vectors"][clean_drift_mask],
                    magnitudes=snapshot["clean_drift_magnitudes"][clean_drift_mask],
                    color=color,
                    name=f"clean transport mode {mode_id}",
                    row=1,
                    col=3,
                    visible=visible,
                    showlegend=False,
                    show_hover=True,
                    arrow_scale=0.32,
                    line_width=1.2,
                    opacity=0.65,
                )
        add_quiver_traces(
            fig,
            points=snapshot["noisy_drift_points"],
            vectors=snapshot["noisy_drift_vectors"],
            magnitudes=snapshot["noisy_drift_magnitudes"],
            color="#444444",
            name="noisy mismatch-gradient field",
            row=1,
            col=2,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=True,
            arrow_scale=0.32,
            line_width=1.2,
            opacity=0.55,
        )
        add_contour_trace(
            fig,
            xs=snapshot["noisy_contour_xs"],
            ys=snapshot["noisy_contour_ys"],
            values=snapshot["noisy_mismatch_norm"],
            row=2,
            col=1,
            visible=visible,
            name="noisy score mismatch norm",
            colorbar_title="||Δscore||",
            hover_label="||Δscore||",
        )
        add_contour_trace(
            fig,
            xs=snapshot["clean_contour_xs"],
            ys=snapshot["clean_contour_ys"],
            values=snapshot["clean_pullback_mismatch_norm"],
            row=2,
            col=2,
            visible=visible,
            name="clean pullback mismatch norm",
            colorbar_title="||Δscore||",
            hover_label="||Δscore||",
        )
        add_contour_trace(
            fig,
            xs=snapshot["drift_xs"],
            ys=snapshot["drift_ys"],
            values=snapshot["clean_drift_norm"],
            row=2,
            col=3,
            visible=visible,
            name="clean transport drift norm",
            colorbar_title="||drift||",
            hover_label="||drift||",
        )
        add_quiver_traces(
            fig,
            points=snapshot["noisy_grid_score_points"],
            vectors=snapshot["noisy_grid_score_vectors"],
            magnitudes=snapshot["noisy_grid_score_magnitudes"],
            color="#111111",
            name="model score field",
            row=2,
            col=1,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=False,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        add_quiver_traces(
            fig,
            points=snapshot["noisy_grid_score_points"],
            vectors=snapshot["noisy_grid_gaussian_score_vectors"],
            magnitudes=snapshot["noisy_grid_gaussian_score_magnitudes"],
            color="#1f77b4",
            name="analytic Gaussian score field",
            row=2,
            col=1,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=False,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        add_model_and_gaussian_score_hover_trace(
            fig,
            points=snapshot["noisy_grid_score_points"],
            model_score_magnitudes=snapshot["noisy_grid_score_magnitudes"],
            gaussian_score_magnitudes=snapshot["noisy_grid_gaussian_score_magnitudes"],
            row=2,
            col=1,
            visible=visible,
        )
        add_quiver_traces(
            fig,
            points=snapshot["clean_grid_difference_points"],
            vectors=snapshot["clean_grid_difference_vectors"],
            magnitudes=snapshot["clean_grid_difference_magnitudes"],
            color="#d62728",
            name="clean pullback transport field",
            row=2,
            col=2,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=True,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        add_quiver_traces(
            fig,
            points=snapshot["drift_grid_points"],
            vectors=snapshot["drift_grid_vectors"],
            magnitudes=snapshot["drift_grid_magnitudes"],
            color="#111111",
            name="noise-averaged transport field",
            row=2,
            col=3,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=True,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.92,
        )
        toggle_trace_indices.append(list(range(start_trace, len(fig.data))))

    contour_trace_start = len(fig.data)
    for row, cols in [(1, [1, 2, 3]), (2, [1, 2, 3])]:
        for col in cols:
            add_latent_contour_traces(
                fig,
                contour_points=contour_points,
                contour_ids=contour_ids,
                contour_levels=contour_levels,
                row=row,
                col=col,
                visible=True,
                showlegend=row == 1 and col == 1,
            )
    contour_trace_indices = list(range(contour_trace_start, len(fig.data)))

    buttons = []
    for snapshot_index, snapshot in enumerate(snapshots):
        visibility = [False] * len(fig.data)
        for trace_index in toggle_trace_indices[snapshot_index]:
            visibility[trace_index] = True
        for trace_index in contour_trace_indices:
            visibility[trace_index] = True
        buttons.append(
            {
                "label": f"t={snapshot['noise_level']:.2f}",
                "method": "update",
                "args": [
                    {
                        "visible": visibility,
                    },
                    {
                        "title": (
                            "direct transport critic diagnostics at step "
                            f"{step} for t={snapshot['noise_level']:.2f}"
                        ),
                        "xaxis4.range": [
                            float(snapshot["noisy_contour_xs"][0].item()),
                            float(snapshot["noisy_contour_xs"][-1].item()),
                        ],
                        "yaxis4.range": [
                            float(snapshot["noisy_contour_ys"][0].item()),
                            float(snapshot["noisy_contour_ys"][-1].item()),
                        ],
                        "xaxis5.range": [
                            float(snapshot["clean_contour_xs"][0].item()),
                            float(snapshot["clean_contour_xs"][-1].item()),
                        ],
                        "yaxis5.range": [
                            float(snapshot["clean_contour_ys"][0].item()),
                            float(snapshot["clean_contour_ys"][-1].item()),
                        ],
                        "xaxis6.range": [
                            float(snapshot["drift_xs"][0].item()),
                            float(snapshot["drift_xs"][-1].item()),
                        ],
                        "yaxis6.range": [
                            float(snapshot["drift_ys"][0].item()),
                            float(snapshot["drift_ys"][-1].item()),
                        ],
                    },
                ],
            }
        )

    for row, col, y_axis in [
        (1, 1, "y"),
        (1, 2, "y2"),
        (1, 3, "y3"),
        (2, 1, "y4"),
        (2, 2, "y5"),
        (2, 3, "y6"),
    ]:
        fig.update_xaxes(
            showgrid=True,
            zeroline=False,
            scaleanchor=y_axis,
            scaleratio=1,
            row=row,
            col=col,
        )
        fig.update_yaxes(
            showgrid=True,
            zeroline=False,
            row=row,
            col=col,
        )

    fig.update_layout(
        title=(
            "direct transport critic diagnostics at step "
            f"{step} for t={experiment_config.plot_t_values[0]:.2f}"
        ),
        height=900,
        width=2080,
        margin={"l": 30, "r": 20, "t": 90, "b": 35},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
        updatemenus=[
            {
                "type": "dropdown",
                "direction": "down",
                "buttons": buttons,
                "x": 0.0,
                "xanchor": "left",
                "y": 1.11,
                "yanchor": "top",
                "showactive": True,
            }
        ],
    )
    return fig


def snapshot_progress(
    show: bool,
) -> dict[str, go.Figure]:
    figures = {
        "training_metrics": plot_training_metrics(),
        "transport_state": plot_transport_state(
            scatter_num_samples=experiment_config.scatter_num_samples,
        ),
        "critic_diagnostics": plot_score_estimates(),
    }
    if show:
        for figure in figures.values():
            figure.show()
    return figures


In [ ]:

def sample_transport_indices(batch_size: int) -> Tensor:
    indices = torch.randint(
        low=0,
        high=transport_points.shape[0],
        size=(batch_size,),
        device=transport_points.device,
    )
    return torch.unique(indices)


def append_log_entry(
    step: int,
    score_matching_loss: float,
    transport_update_norm: float,
) -> None:
    mean_error, cov_error = transport_moment_errors(transport_points)
    logs["steps"].append(step)
    logs["score_matching"].append(score_matching_loss)
    logs["transport_update_norm"].append(transport_update_norm)
    logs["mean_error"].append(mean_error)
    logs["cov_error"].append(cov_error)


def run_direct_transport(show_snapshots: bool) -> None:
    progress_bar = tqdm(
        range(1, experiment_config.total_steps + 1),
        desc="direct transport",
        unit="step",
    )
    for step in progress_bar:
        model.train()
        batch_indices = sample_transport_indices(experiment_config.train_batch_size)
        clean_batch = transport_points[batch_indices].detach()
        optimizer.zero_grad(set_to_none=True)

        t = sample_noise_levels(
            clean_batch.shape[0],
            device=clean_batch.device,
            dtype=clean_batch.dtype,
        )
        noise = torch.randn_like(clean_batch)
        with training_device_context():
            noisy_batch = (1.0 - t) * clean_batch + t * noise
            predicted_noise = model.noise_critic(noisy_batch, t=t)
            score_matching_loss = (
                F.huber_loss(
                    predicted_noise,
                    noise,
                    delta=5.0,
                    reduction="none",
                )
                .sum(dim=1)
                .mean()
            )
        score_matching_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.noise_critic.parameters(),
            max_norm=experiment_config.grad_clip_norm,
        )
        optimizer.step()

        transport_update_norm = 0.0
        if step >= experiment_config.warmup_steps:
            drift_t = sample_noise_levels(
                clean_batch.shape[0],
                device=clean_batch.device,
                dtype=clean_batch.dtype,
            )
            drift_noise = torch.randn_like(clean_batch)
            _, _, clean_drift = score_difference_transport_vectors(
                clean_latents=clean_batch,
                t=drift_t,
                noise=drift_noise,
            )
            clipped_drift, _ = clip_vector_norms(
                clean_drift,
                max_norm=experiment_config.max_transport_update_norm,
            )
            applied_update = experiment_config.transport_step_size * clipped_drift
            updated_points = clean_batch + applied_update
            transport_points.index_copy_(0, batch_indices, updated_points.detach())
            transport_update_norm = float(applied_update.norm(dim=1).mean().item())

        should_log = (
            step == 1
            or step == experiment_config.warmup_steps
            or step % experiment_config.log_every_steps == 0
            or step % experiment_config.snapshot_every_steps == 0
            or step == experiment_config.total_steps
        )
        should_snapshot = (
            step == 1
            or step == experiment_config.warmup_steps
            or step % experiment_config.snapshot_every_steps == 0
            or step == experiment_config.total_steps
        )
        if not should_log:
            continue

        append_log_entry(
            step=step,
            score_matching_loss=float(score_matching_loss.detach().item()),
            transport_update_norm=transport_update_norm,
        )
        progress_bar.set_postfix(
            score=f"{logs['score_matching'][-1]:.4f}",
            drift=f"{logs['transport_update_norm'][-1]:.4f}",
            mean=f"{logs['mean_error'][-1]:.4f}",
            cov=f"{logs['cov_error'][-1]:.4f}",
        )

        if should_snapshot:
            snapshot_progress(show=show_snapshots)


snapshot_progress(show=True)
run_direct_transport(show_snapshots=True)
